# oauth_flow

**One-time interactive notebook** — generates the Spotify authorization URL,
exchanges the callback code for tokens, and prints the `refresh_token` to
store in Databricks secrets.

Run manually once per Spotify account. Do **not** include in automated pipelines.

Dependencies:
- `%run ../config/settings`
- `%run ../auth/token_manager`

In [ ]:
# %run ../config/settings
# %run ../auth/token_manager

## Step 1 — Get the authorization URL

Run the cell, open the printed URL in your browser, grant access,
then copy the full callback URL from the browser address bar.

In [ ]:
auth = SpotifyAuth(
    client_id     = dbutils.secrets.get(SECRET_SCOPE, "spotify_client_id"),
    client_secret = dbutils.secrets.get(SECRET_SCOPE, "spotify_client_secret"),
    redirect_uri  = dbutils.secrets.get(SECRET_SCOPE, "spotify_redirect_uri"),
)

print("Open this URL in your browser and grant access:")
print()
print(auth.get_authorization_url())

## Step 2 — Exchange the authorization code

Paste the full callback URL (e.g. `http://localhost/?code=AQD...`) below.

In [ ]:
import urllib.parse

callback_url = ""  # <-- paste the full callback URL here

parsed = urllib.parse.urlparse(callback_url)
code   = urllib.parse.parse_qs(parsed.query)["code"][0]

auth.exchange_code(code)

print("Access token (preview):", (auth._access_token or "")[:30], "...")
print()
print("Refresh token (store this in Databricks secrets):")
print(auth.refresh_token)

## Step 3 — Persist the refresh token

Run from a terminal or Databricks CLI:

```bash
databricks secrets put-secret spotify_secrets spotify_refresh_token
```

Paste the refresh token printed above when prompted.